In [3]:
import os

os.chdir(r"c:\Users\Sufiyan Asif\OneDrive\Desktop\Deep_Learning_Project1\DEEP_Learning_Project1")

print(os.getcwd())

c:\Users\Sufiyan Asif\OneDrive\Desktop\Deep_Learning_Project1\DEEP_Learning_Project1


In [4]:
import dagshub
dagshub.init(repo_owner='stopmold8290', repo_name='DEEP_Learning_Project1', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as stopmold8290

Initialized MLflow to track repo "stopmold8290/DEEP_Learning_Project1"

Repository stopmold8290/DEEP_Learning_Project1 initialized!

c:\Users\Sufiyan Asif\OneDrive\Desktop\Deep_Learning_Project1\DEEP_Learning_Project1\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏃 View run casual-hound-394 at: https://dagshub.com/stopmold8290/DEEP_Learning_Project1.mlflow/#/experiments/0/runs/7ee7068e6673411f8c7e679e26cedaac
🧪 View experiment at: https://dagshub.com/stopmold8290/DEEP_Learning_Project1.mlflow/#/experiments/0


In [5]:
import tensorflow as tf

In [6]:
model  = tf.keras.models.load_model("artifacts/training/trained_model.h5")

In [7]:
from pathlib import Path

path = Path("artifacts/training/trained_model.h5")

print(path.exists())

True


In [8]:
import os

print(os.getcwd())

c:\Users\Sufiyan Asif\OneDrive\Desktop\Deep_Learning_Project1\DEEP_Learning_Project1


In [9]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [10]:
from cnn_classifier.constant import *
from cnn_classifier.utils.common import read_yaml, create_directories, save_json

In [11]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class EvaluationConfig:

    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int


class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARMAS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)

        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:

        eval_config = EvaluationConfig(
            path_of_model=Path(
                "artifacts/training/trained_model.h5"
            ),

            training_data=Path(
                "artifacts/data_ingestion/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
            ),

            all_params=dict(self.params),
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )

        return eval_config

In [12]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [13]:
class Evaluation:

    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1. / 255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:

        return tf.keras.models.load_model(path)

    def evaluation(self):

        self.model = self.load_model(
            self.config.path_of_model
        )

        self._valid_generator()

        self.score = self.model.evaluate(
            self.valid_generator
        )

        self.save_score()

    def save_score(self):

        scores = {
            "loss": float(self.score[0]),
            "accuracy": float(self.score[1])
        }

        save_json(
            path=Path("score.json"),
            data=scores
        )

    def log_into_mlflow(self):

        with mlflow.start_run():

            mlflow.log_params(
                dict(self.config.all_params)
            )

            mlflow.log_metrics({
                "loss": float(self.score[0]),
                "accuracy": float(self.score[1])
            })

            mlflow.keras.log_model(
                self.model,
                "model"
            )

In [14]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(config= eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
    
except Exception as e:
    raise e

Found 2207 images belonging to 2 classes.
138/138 ━━━━━━━━━━━━━━━━━━━━ 2092s 15s/step - accuracy: 0.8441 - loss: 0.4596


2026/05/14 09:32:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/14 09:32:08 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


🏃 View run unruly-donkey-986 at: https://dagshub.com/stopmold8290/DEEP_Learning_Project1.mlflow/#/experiments/0/runs/983a1cdc4be546bda2a84390abe48e92
🧪 View experiment at: https://dagshub.com/stopmold8290/DEEP_Learning_Project1.mlflow/#/experiments/0
